In [1]:
!pip install line_profiler

In [2]:
%load_ext line_profiler

In [3]:
import random

def create_grid(rows, cols, alive):
    grid = [[0] * (cols + 2) for _ in range(rows + 2)]
    count = round(rows * cols * alive)
    rows += 1
    cols += 1
    while count:
        row, col = random.randrange(1, rows), random.randrange(1, cols)
        if not grid[row][col]:
            grid[row][col] = 1
            count -= 1
    return grid

In [4]:
%lprun -f create_grid grid = create_grid(500, 500, 0.5)

Timer unit: 1e-07 s

Total time: 2.4241 s
File: C:\Users\Serkan\AppData\Local\Temp\ipykernel_7580\4072744524.py
Function: create_grid at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def create_grid(rows, cols, alive):
     4         1      10398.0  10398.0      0.0      grid = [[0] * (cols + 2) for _ in range(rows + 2)]
     5         1         32.0     32.0      0.0      count = round(rows * cols * alive)
     6         1          7.0      7.0      0.0      rows += 1
     7         1          5.0      5.0      0.0      cols += 1
     8    173459     738279.0      4.3      3.0      while count:
     9    173458   21474564.0    123.8     88.6          row, col = random.randrange(1, rows), random.randrange(1, cols)
    10    173458     933927.0      5.4      3.9          if not grid[row][col]:
    11    125000     546638.0      4.4      2.3              grid[row][col] = 1
    12    125000     537109.0      4.3    

In [5]:
%%timeit -n 5 -r 3
grid = create_grid(500, 500, 0.5)

216 ms ± 1.21 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [6]:
def save_grid(grid, filename):
    with open(filename, "w") as file:
        rows, cols = len(grid) - 2, len(grid[0]) - 2
        file.write(f"{rows},{cols}\n")
        for y, row in enumerate(grid):
            for x, cell in enumerate(row):
                if cell:
                    file.write(f"{y - 1},{x - 1}\n")

In [7]:
%lprun -f save_grid save_grid(grid, "input_500x500_0.5.txt")

Timer unit: 1e-07 s

Total time: 0.451692 s
File: C:\Users\Serkan\AppData\Local\Temp\ipykernel_7580\1588092382.py
Function: save_grid at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def save_grid(grid, filename):
     2         2       3462.0   1731.0      0.1      with open(filename, "w") as file:
     3         1         25.0     25.0      0.0          rows, cols = len(grid) - 2, len(grid[0]) - 2
     4         1         95.0     95.0      0.0          file.write(f"{rows},{cols}\n")
     5       503       2784.0      5.5      0.1          for y, row in enumerate(grid):
     6    252506    1148152.0      4.5     25.4              for x, cell in enumerate(row):
     7    252004    1006773.0      4.0     22.3                  if cell:
     8    125000    2355625.0     18.8     52.2                      file.write(f"{y - 1},{x - 1}\n")

In [8]:
%%timeit -n 5 -r 3
save_grid(grid, "input_500x500_0.5.txt")

92.6 ms ± 186 μs per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [9]:
def read_grid(filename):
    with open(filename) as file:
        rows, cols = map(int, file.readline().split(','))
        grid = [[0] * (cols + 2) for _ in range(rows + 2)]
        for line in file:
            row, col = line.split(',')
            grid[int(row) + 1][int(col) + 1] = 1
    return grid

In [10]:
%lprun -f read_grid grid = read_grid("input_500x500_0.5.txt")

Timer unit: 1e-07 s

Total time: 0.230159 s
File: C:\Users\Serkan\AppData\Local\Temp\ipykernel_7580\2820859252.py
Function: read_grid at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def read_grid(filename):
     2         2      85182.0  42591.0      3.7      with open(filename) as file:
     3         1        700.0    700.0      0.0          rows, cols = map(int, file.readline().split(','))
     4         1      27996.0  27996.0      1.2          grid = [[0] * (cols + 2) for _ in range(rows + 2)]
     5    125001     650936.0      5.2     28.3          for line in file:
     6    125000     704091.0      5.6     30.6              row, col = line.split(',')
     7    125000     832670.0      6.7     36.2              grid[int(row) + 1][int(col) + 1] = 1
     8         1         16.0     16.0      0.0      return grid

In [11]:
%%timeit -n 5 -r 3
grid = read_grid("input_500x500_0.5.txt")

44.4 ms ± 193 μs per loop (mean ± std. dev. of 3 runs, 5 loops each)


- Any **live cell with fewer than two live neighbours dies**, as if by underpopulation.
- Any **live cell with two or three live neighbours lives** on to the next generation.
- Any **live cell with more than three live neighbours dies**, as if by overpopulation.
- Any **dead cell with exactly three live neighbours becomes a live cell**, as if by reproduction.

In [12]:
def tick(grid):
    rows, cols = len(grid), len(grid[0])
    new_grid = [[0] * cols for _ in range(rows)]
    for row in range(1, rows - 1):
        rm, rp = row - 1, row + 1
        for col in range(1, cols - 1):
            cm, cp = col - 1, col + 1
            count = grid[rm][cm] + grid[rm][col] + grid[rm][cp] + grid[row][cm] + grid[row][cp] + grid[rp][cm] + grid[rp][col] + grid[rp][cp]
            if count == 3 or (grid[row][col] == 1 and count == 2):
                new_grid[row][col] = 1
    return new_grid

In [13]:
%lprun -f tick next_grid = tick(grid)

Timer unit: 1e-07 s

Total time: 0.500718 s
File: C:\Users\Serkan\AppData\Local\Temp\ipykernel_7580\1554368824.py
Function: tick at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def tick(grid):
     2         1         18.0     18.0      0.0      rows, cols = len(grid), len(grid[0])
     3         1      12561.0  12561.0      0.3      new_grid = [[0] * cols for _ in range(rows)]
     4       501       2080.0      4.2      0.0      for row in range(1, rows - 1):
     5       500       2217.0      4.4      0.0          rm, rp = row - 1, row + 1
     6    250500    1061388.0      4.2     21.2          for col in range(1, cols - 1):
     7    250000    1074600.0      4.3     21.5              cm, cp = col - 1, col + 1
     8    250000    1419713.0      5.7     28.4              count = grid[rm][cm] + grid[rm][col] + grid[rm][cp] + grid[row][cm] + grid[row][cp] + grid[rp][cm] + grid[rp][col] + grid[rp][cp]
     9    2

In [14]:
%%timeit -n 5 -r 3
next_grid = tick(grid)

78 ms ± 453 μs per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [15]:
grid = read_grid("input_5x5.txt")
next_grid = tick(grid)
next_grid

[[0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 0],
 [0, 0, 1, 1, 1, 0, 0],
 [0, 0, 1, 1, 1, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0]]

In [16]:
next_grid = tick(next_grid)
next_grid

[[0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 0],
 [0, 1, 0, 0, 1, 0, 0],
 [0, 0, 1, 0, 1, 0, 0],
 [0, 0, 0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0]]